```markdown
# Consolidated Machine Learning Pipeline
This notebook loads preprocessed data and systematically trains, fine-tunes, and compares performance for **Decision Tree**, **Random Forest**, and **XGBoost** models.
```

In [21]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# 1. Load Preprocessed Datasets
train = pd.read_csv("/content/train_preprocessed.csv")
val = pd.read_csv("/content/validation_preprocessed.csv")
test = pd.read_csv("/content/test_preprocessed.csv")

TARGET = "passengers"

# Extract numeric columns and handle alignment
X_train = train.drop(columns=[TARGET]).select_dtypes(include=np.number).copy()
y_train = train[TARGET]

X_val = val.drop(columns=[TARGET]).select_dtypes(include=np.number).copy()
y_val = val[TARGET]

X_test = test.drop(columns=[TARGET]).select_dtypes(include=np.number).copy()
y_test = test[TARGET]

X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# Impute any missing values with training median
medians = X_train.median()
X_train = X_train.fillna(medians)
X_val = X_val.fillna(medians)
X_test = X_test.fillna(medians)

print(f"Data Shapes - Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Data Shapes - Train: (21276, 11), Val: (7164, 11), Test: (7272, 11)


In [22]:
# 2. Setup Evaluation metrics accumulator
results = []

def evaluate_model(model, X, y, dataset_name, model_name):
    pred = model.predict(X)
    r2 = r2_score(y, pred)
    rmse = np.sqrt(mean_squared_error(y, pred))
    mae = mean_absolute_error(y, pred)
    results.append({
        "Model": model_name,
        "Dataset": dataset_name,
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae
    })
    print(f"[{model_name}] {dataset_name} -> R2: {r2:.4f}, RMSE: {rmse:.4f}, MAE: {mae:.4f}")

```markdown
## 3. Decision Tree (Normal & Tuned)
```

In [23]:
# Tuned Decision Tree
dt_params = {
    "max_depth": [4, 6, 8, 10, 12, 15, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": [0.8, 1.0, None],
    "ccp_alpha": [0.0, 0.001, 0.01]
}

dt_search = RandomizedSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_distributions=dt_params,
    n_iter=20,
    scoring="r2",
    cv=TimeSeriesSplit(n_splits=3),
    random_state=42,
    n_jobs=-1
)
dt_search.fit(X_train, y_train)
dt_tuned = dt_search.best_estimator_
print("Best DT Parameters:", dt_search.best_params_)

evaluate_model(dt_tuned, X_train, y_train, "Train", "Decision Tree Tuned")
evaluate_model(dt_tuned, X_val, y_val, "Val", "Decision Tree Tuned")
evaluate_model(dt_tuned, X_test, y_test, "Test", "Decision Tree Tuned")

Best DT Parameters: {'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 1.0, 'max_depth': 8, 'ccp_alpha': 0.01}
[Decision Tree Tuned] Train -> R2: 0.5451, RMSE: 8.0246, MAE: 6.1784
[Decision Tree Tuned] Val -> R2: 0.5388, RMSE: 8.0569, MAE: 6.2152
[Decision Tree Tuned] Test -> R2: 0.5445, RMSE: 8.0945, MAE: 6.1683


```markdown
## 4. Random Forest (Normal & Tuned)
```

In [24]:
# Normal Random Forest
rf_normal = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_normal.fit(X_train, y_train)

evaluate_model(rf_normal, X_train, y_train, "Train", "Random Forest Normal")
evaluate_model(rf_normal, X_val, y_val, "Val", "Random Forest Normal")
evaluate_model(rf_normal, X_test, y_test, "Test", "Random Forest Normal")

[Random Forest Normal] Train -> R2: 0.9266, RMSE: 3.2232, MAE: 2.4120
[Random Forest Normal] Val -> R2: 0.9254, RMSE: 3.2400, MAE: 2.4174
[Random Forest Normal] Test -> R2: 0.9265, RMSE: 3.2524, MAE: 2.4209


In [25]:
# Tuned Random Forest
rf_params = {
    "n_estimators": [100, 200],
    "max_depth": [8, 12, 16, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", 0.8, 1.0]
}

rf_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=1),
    param_distributions=rf_params,
    n_iter=10,
    scoring="r2",
    cv=TimeSeriesSplit(n_splits=3),
    random_state=42,
    n_jobs=-1
)
rf_search.fit(X_train, y_train)
rf_tuned = rf_search.best_estimator_
print("Best RF Parameters:", rf_search.best_params_)

evaluate_model(rf_tuned, X_train, y_train, "Train", "Random Forest Tuned")
evaluate_model(rf_tuned, X_val, y_val, "Val", "Random Forest Tuned")
evaluate_model(rf_tuned, X_test, y_test, "Test", "Random Forest Tuned")

Best RF Parameters: {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'max_depth': None}
[Random Forest Tuned] Train -> R2: 0.7007, RMSE: 6.5093, MAE: 4.9137
[Random Forest Tuned] Val -> R2: 0.6990, RMSE: 6.5088, MAE: 4.9268
[Random Forest Tuned] Test -> R2: 0.6966, RMSE: 6.6064, MAE: 4.9239


In [29]:
# Optimizing Random Forest with different, expanded hyperparameters to improve R2
rf_improved_params = {
    "n_estimators": [150, 250, 300],
    "max_depth": [15, 20, 30, None],
    "min_samples_split": [2, 4, 6],
    "min_samples_leaf": [1, 2],
    "max_features": [0.7, 0.9, "sqrt"]
}

rf_improved_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=rf_improved_params,
    n_iter=15,
    scoring="r2",
    cv=TimeSeriesSplit(n_splits=3),
    random_state=42,
    n_jobs=-1
)
rf_improved_search.fit(X_train, y_train)
rf_improved_tuned = rf_improved_search.best_estimator_
print("Best Improved RF Parameters:", rf_improved_search.best_params_)

evaluate_model(rf_improved_tuned, X_train, y_train, "Train", "Random Forest Improved Tuned")
evaluate_model(rf_improved_tuned, X_val, y_val, "Val", "Random Forest Improved Tuned")
evaluate_model(rf_improved_tuned, X_test, y_test, "Test", "Random Forest Improved Tuned")

/usr/local/lib/python3.13/dist-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best Improved RF Parameters: {'n_estimators': 150, 'min_samples_split': 6, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': None}
[Random Forest Improved Tuned] Train -> R2: 0.7765, RMSE: 5.6250, MAE: 4.2069
[Random Forest Improved Tuned] Val -> R2: 0.7753, RMSE: 5.6237, MAE: 4.2101
[Random Forest Improved Tuned] Test -> R2: 0.7736, RMSE: 5.7070, MAE: 4.2158


In [30]:
# Advanced Random Forest tuning targeting high R2 score by allowing full feature sets
rf_advanced_params = {
    "n_estimators": [200, 300, 400],
    "max_depth": [25, 35, None],
    "min_samples_split": [2, 4],
    "min_samples_leaf": [1, 2],
    "max_features": [0.8, 0.9, 1.0]  # Higher feature ratios to match/beat the default configuration
}

rf_advanced_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=rf_advanced_params,
    n_iter=10,
    scoring="r2",
    cv=TimeSeriesSplit(n_splits=3),
    random_state=42,
    n_jobs=-1
)
rf_advanced_search.fit(X_train, y_train)
rf_advanced_tuned = rf_advanced_search.best_estimator_
print("Best Advanced RF Parameters:", rf_advanced_search.best_params_)

evaluate_model(rf_advanced_tuned, X_train, y_train, "Train", "Random Forest Advanced Tuned")
evaluate_model(rf_advanced_tuned, X_val, y_val, "Val", "Random Forest Advanced Tuned")
evaluate_model(rf_advanced_tuned, X_test, y_test, "Test", "Random Forest Advanced Tuned")

Best Advanced RF Parameters: {'n_estimators': 400, 'min_samples_split': 4, 'min_samples_leaf': 2, 'max_features': 0.8, 'max_depth': 25}
[Random Forest Advanced Tuned] Train -> R2: 0.8425, RMSE: 4.7219, MAE: 3.4237
[Random Forest Advanced Tuned] Val -> R2: 0.8421, RMSE: 4.7141, MAE: 3.4252
[Random Forest Advanced Tuned] Test -> R2: 0.8393, RMSE: 4.8083, MAE: 3.4427


```markdown
## 5. XGBoost (Normal & Tuned)
```

In [26]:
# Normal XGBoost
xgb_normal = XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)
xgb_normal.fit(X_train, y_train)

evaluate_model(xgb_normal, X_train, y_train, "Train", "XGBoost Normal")
evaluate_model(xgb_normal, X_val, y_val, "Val", "XGBoost Normal")
evaluate_model(xgb_normal, X_test, y_test, "Test", "XGBoost Normal")

[XGBoost Normal] Train -> R2: 0.6668, RMSE: 6.8679, MAE: 5.2166
[XGBoost Normal] Val -> R2: 0.6599, RMSE: 6.9188, MAE: 5.2511
[XGBoost Normal] Test -> R2: 0.6647, RMSE: 6.9452, MAE: 5.2231


In [27]:
# Tuned XGBoost
xgb_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7, 9],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.9, 1.0]
}

xgb_search = RandomizedSearchCV(
    XGBRegressor(random_state=42, n_jobs=-1),
    param_distributions=xgb_params,
    n_iter=15,
    scoring="r2",
    cv=TimeSeriesSplit(n_splits=3),
    random_state=42,
    n_jobs=-1
)
xgb_search.fit(X_train, y_train)
xgb_tuned = xgb_search.best_estimator_
print("Best XGB Parameters:", xgb_search.best_params_)

evaluate_model(xgb_tuned, X_train, y_train, "Train", "XGBoost Tuned")
evaluate_model(xgb_tuned, X_val, y_val, "Val", "XGBoost Tuned")
evaluate_model(xgb_tuned, X_test, y_test, "Test", "XGBoost Tuned")

Best XGB Parameters: {'subsample': 0.9, 'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.05, 'colsample_bytree': 1.0}
[XGBoost Tuned] Train -> R2: 0.5594, RMSE: 7.8974, MAE: 6.0669
[XGBoost Tuned] Val -> R2: 0.5578, RMSE: 7.8893, MAE: 6.0634
[XGBoost Tuned] Test -> R2: 0.5553, RMSE: 7.9982, MAE: 6.0744


```markdown
## 6. Performance Summary Table
```

In [28]:
# Format and display the evaluation metrics comparison side-by-side
results_df = pd.DataFrame(results)
display(results_df.pivot(index="Model", columns="Dataset", values=["R2", "RMSE", "MAE"]))

R2                          RMSE            \
Dataset                   Test     Train       Val      Test     Train   
Model                                                                    
Decision Tree Tuned   0.544483  0.545126  0.538810  8.094483  8.024567   
Random Forest Normal  0.926458  0.926613  0.925419  3.252396  3.223197   
Random Forest Tuned   0.696572  0.700695  0.699013  6.606399  6.509280   
XGBoost Normal        0.664655  0.666811  0.659905  6.945165  6.867857   
XGBoost Tuned         0.555252  0.559424  0.557802  7.998226  7.897443   

                                     MAE                      
Dataset                    Val      Test     Train       Val  
Model                                                         
Decision Tree Tuned   8.056918  6.168320  6.178357  6.215172  
Random Forest Normal  3.239977  2.420903  2.411955  2.417415  
Random Forest Tuned   6.508823  4.923904  4.913684  4.926825  
XGBoost Normal        6.918777  5.223070  5.216636  5.251145  
XGBoost Tuned         7.889279  6.074359  6.066887  6.063435